In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

# Configuração visual para os gráficos ficarem com cara de painel profissional
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [5]:
import pandas as pd
import os

# 1. Tentar encontrar o arquivo novamente
caminhos = [
    "../data/processed/metropt3.parquet",
    "data/processed/metropt3.parquet",
    "./data/processed/metropt3.parquet"
]

df = None
for p in caminhos:
    if os.path.exists(p):
        df = pd.read_parquet(p)
        break

if df is not None:
    print(f"📊 Análise de 'Pureza' das Colunas (Frequência do valor mais comum)\n" + "-"*60)
    
    resultados = []
    for col in df.columns:
        if col in ['Unnamed: 0', 'timestamp']: continue
        
        # Pega o valor mais frequente e sua contagem
        contagem = df[col].value_counts()
        top_valor = contagem.index[0]
        top_freq = contagem.iloc[0]
        
        # Calcula a porcentagem
        porcentagem = (top_freq / len(df)) * 100
        qtd_unicos = df[col].nunique()
        
        resultados.append({
            "Coluna": col,
            "Valor mais comum": top_valor,
            "Porcentagem": porcentagem,
            "Valores Únicos": qtd_unicos
        })

    # Transforma em DataFrame para ordenar e mostrar bonito
    df_analise = pd.DataFrame(resultados).sort_values(by="Porcentagem", ascending=False)
    
    for _, row in df_analise.iterrows():
        alerta = "⚠️ [QUASE CONSTANTE]" if 99.9 < row['Porcentagem'] < 100 else ""
        if row['Porcentagem'] == 100: alerta = "❌ [CONSTANTE - DESCARTAR]"
        
        print(f"{row['Coluna'].ljust(18)} | {row['Porcentagem']:6.2f}% é o valor '{row['Valor mais comum']}' | Unicos: {row['Valores Únicos']} {alerta}")

else:
    print("Não foi possível carregar o arquivo para análise.")

📊 Análise de 'Pureza' das Colunas (Frequência do valor mais comum)
------------------------------------------------------------
LPS                |  99.66% é o valor '0.0' | Unicos: 2 
Pressure_switch    |  99.14% é o valor '1.0' | Unicos: 2 
Caudal_impulses    |  93.71% é o valor '1.0' | Unicos: 2 
Towers             |  91.98% é o valor '1.0' | Unicos: 2 
Oil_level          |  90.42% é o valor '1.0' | Unicos: 2 
DV_eletric         |  83.94% é o valor '0.0' | Unicos: 2 
COMP               |  83.70% é o valor '1.0' | Unicos: 2 
MPG                |  83.27% é o valor '1.0' | Unicos: 2 
TP2                |  29.37% é o valor '-0.012000000104308128' | Unicos: 5257 
DV_pressure        |  21.24% é o valor '-0.019999999552965164' | Unicos: 2257 
Motor_current      |  21.08% é o valor '0.042500000447034836' | Unicos: 1809 
H1                 |   2.92% é o valor '-0.014000000432133675' | Unicos: 2665 
Oil_temperature    |   1.39% é o valor '65.1500015258789' | Unicos: 2462 
Reservoirs         

In [2]:
import pandas as pd

# 1. Garantir que a coluna timestamp seja tratada como Data/Hora pelo Python
print("Convertendo timestamp para datetime (pode levar alguns segundos)...")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Criar a coluna 'anomaly' começando com 0 (Tudo Normal)
df['anomaly'] = 0

# 3. Mapear as 4 falhas documentadas no PDF
falhas = [
    ("2020-04-18 00:00:00", "2020-04-18 23:59:00"), # Falha #1
    ("2020-05-29 23:30:00", "2020-05-30 06:00:00"), # Falha #2
    ("2020-06-05 10:00:00", "2020-06-07 14:30:00"), # Falha #3
    ("2020-07-15 14:30:00", "2020-07-15 19:00:00")  # Falha #4
]

# 4. Preencher com 1 os horários em que o compressor estava quebrado
print("Marcando as falhas documentadas...")
for inicio, fim in falhas:
    mask = (df['timestamp'] >= inicio) & (df['timestamp'] <= fim)
    df.loc[mask, 'anomaly'] = 1

# 5. Calcular a porcentagem real de falhas
total_falhas = df['anomaly'].sum()
taxa_falha = (total_falhas / len(df)) * 100

print("-" * 30)
print(f"✅ Coluna 'anomaly' criada com sucesso!")
print(f"Total de registros de falha (minutos quebrado): {total_falhas}")
print(f"Taxa de falha no dataset: {taxa_falha:.4f}%")
print("-" * 30)

# Mostra as colunas novamente, agora com a 'anomaly' no final
print(df.columns.tolist())

Convertendo timestamp para datetime (pode levar alguns segundos)...


NameError: name 'df' is not defined

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

# 1. Preparando os dados (tirando colunas de texto/data e a própria resposta)
# 'Unnamed: 0' é só o índice do arquivo, não serve pra IA
cols_to_drop = ['Unnamed: 0', 'timestamp', 'anomaly']
X = df.drop(columns=cols_to_drop)
y = df['anomaly']

# ==========================================
# GRÁFICO 1: CORRELAÇÃO (O que sobe quando a falha chega?)
# ==========================================
plt.figure(figsize=(10, 8))
# Calcula a correlação apenas com a coluna de anomalia
corr = df.drop(columns=['Unnamed: 0', 'timestamp']).corr()[['anomaly']].sort_values(by='anomaly', ascending=False)

sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlação dos Sensores com a Falha (Air Leak)", fontsize=14)
plt.show()

# ==========================================
# GRÁFICO 2: FEATURE IMPORTANCE (A Visão da IA)
# ==========================================
print("\nTreinando um modelo rápido para ver a importância dos sensores...")
# Usamos max_depth=5 e poucas árvores só para ser rápido na análise
rf_test = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
rf_test.fit(X, y)

# Extraindo e ordenando a importância
importancias = pd.Series(rf_test.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=importancias.values, y=importancias.index, palette='viridis')
plt.title("Top Sensores Mais Importantes para Prever a Falha", fontsize=14)
plt.xlabel("Grau de Importância (%)", fontsize=12)
plt.ylabel("Sensores", fontsize=12)

# Adicionando os valores nas barras
for index, value in enumerate(importancias.values):
    plt.text(value, index, f"{value:.3f}", va='center')

plt.show()

In [ ]:
# Calcula o percentual de dados nulos por coluna
missing_data = df.isnull().mean() * 100
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if missing_data.empty:
    print("✅ Excelente notícia! Não há dados faltantes (NaN) no seu dataset.")
else:
    print("⚠️ Atenção: Encontramos dados faltantes!")
    plt.figure(figsize=(10, 5))
    sns.barplot(x=missing_data.values, y=missing_data.index, palette="Reds_r")
    plt.title("Percentual de Dados Faltantes por Sensor")
    plt.xlabel("% Faltante")
    plt.show()